In [71]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score,RandomizedSearchCV
from sklearn.metrics import make_scorer,cohen_kappa_score,matthews_corrcoef,precision_score,accuracy_score,recall_score,f1_score,roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE,ADASYN
from imblearn.pipeline import Pipeline
from scipy.stats import friedmanchisquare
from scipy.stats import rankdata

In [57]:
data=pd.read_csv("bank-additional-full.csv",sep=';')

In [58]:
data=data.drop("duration",axis=1)

In [59]:
allColumns=data.select_dtypes(include='object').columns
for col in allColumns:
  data[col]=data[col].replace('unknown',data[col].mode()[0])

In [60]:
data['y']=data['y'].map({'yes':1,'no':0})

In [61]:
data=pd.get_dummies(data,drop_first=True)

In [62]:
X=data.drop('y',axis=1)
y=data['y']

In [63]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [64]:
scalar=StandardScaler()
scalar.fit_transform(X_train)
scalar.transform(X_test)

array([[-0.77033007,  0.88631588,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [-0.28972159, -0.56702251,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [ 3.17065947, -0.20368791,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       ...,
       [-0.67420837, -0.56702251,  0.19658384, ..., -0.4964409 ,
        -2.50346033, -0.18627755],
       [ 0.38313029,  1.61298507,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [ 0.19088689,  0.88631588,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755]])

In [65]:
kappa=make_scorer(cohen_kappa_score)
mcc=make_scorer(matthews_corrcoef)
scoring={
    'accuracy':'accuracy',
    'precision':'precision',
    'recall':'recall',
    'f1':'f1',
    'roc_auc':'roc_auc',
    'kappa':kappa,
    'mcc':mcc
}

In [66]:
smote=SMOTE(random_state=42)
adasyn=ADASYN(random_state=42)

In [67]:
models = {
    "Decision Tree": (DecisionTreeClassifier(random_state=42),{'model__max_depth': [5, 10, 15],'model__min_samples_split': [2, 5]}),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            'model__n_estimators': [50, 100],
            'model__max_depth': [5, 10]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            'model__n_neighbors': [3, 5, 7]
        }
    ),

    "Logistic Regression": (
        LogisticRegression(max_iter=1000, random_state=42),
        {
            'model__C': [0.01, 0.1, 1]
        }
    ),

    "XGBoost": (
        XGBClassifier(
            eval_metric='logloss',
            random_state=42,
            verbosity=0
        ),
        {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5]
        }
    ),

    "CatBoost": (
        CatBoostClassifier(
            verbose=0,
            random_state=42
        ),
        {
            'model__iterations': [50, 100],
            'model__depth': [4, 6]
        }
    )
}

In [68]:
samplers = {

    "Baseline": 'passthrough',

    "SMOTE": SMOTE(random_state=42),

    "ADASYN": ADASYN(random_state=42)
}

In [69]:

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [72]:

all_results = []

friedman_scores = {}
friedman_mean_scores = {}
for sampling_name, sampler in samplers.items():

    print(f"SAMPLING : {sampling_name}")
    for model_name, (model, params) in models.items():

        print(f"\Model : {model_name}")

        pipeline = Pipeline([
            ('sampler', sampler),
            ('model', model)
        ])
        random_search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=params,
            n_iter=3,
            scoring='f1',
            cv=3,
            random_state=42,
            n_jobs=-1
        )

        random_search.fit(X_train, y_train)
        best_model = random_search.best_estimator_

        cv_scores = cross_val_score(
            best_model,
            X_train,
            y_train,
            cv=skf,
            scoring='recall'
        )
        mean_cv_score = np.mean(cv_scores)

        print("Best recall Score :", round(mean_cv_score, 4))
        best_model.fit(X_train, y_train)

        y_pred = best_model.predict(X_test)
        y_prob = best_model.predict_proba(X_test)[:, 1]
        accuracy = accuracy_score(y_test, y_pred)

        precision = precision_score(y_test, y_pred)

        recall = recall_score(y_test, y_pred)

        f1 = f1_score(y_test, y_pred)

        roc_auc = roc_auc_score(y_test, y_prob)
        print("Test recall :", round(recall, 4))

        #storing all results of models to generate excel file
        all_results.append({
            'Sampling': sampling_name,
            'Model': model_name,
            'CV recall Score': round(mean_cv_score, 4),
            'Accuracy': round(accuracy, 4),
            'Precision': round(precision, 4),
            'Recall': round(recall, 4),
            'F1 Score': round(f1, 4),
            'ROC AUC': round(roc_auc, 4),
            'Best Parameters': str(random_search.best_params_)
        })
        key = f"{sampling_name} - {model_name}"

        friedman_scores[key] = cv_scores

        friedman_mean_scores[key] = mean_cv_score

SAMPLING : Baseline
\Model : Decision Tree


<>:10: SyntaxWarning: invalid escape sequence '\M'
<>:10: SyntaxWarning: invalid escape sequence '\M'
/tmp/ipykernel_10782/3976042654.py:10: SyntaxWarning: invalid escape sequence '\M'
  print(f"\Model : {model_name}")


Best recall Score : 0.2619
Test recall : 0.2478
\Model : Random Forest
Best recall Score : 0.2128
Test recall : 0.222
\Model : KNN
Best recall Score : 0.2756
Test recall : 0.2953
\Model : Logistic Regression


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Best recall Score : 0.2198


...

Test recall : 0.4935
\Model : XGBoost
Best recall Score : 0.4661
Test recall : 0.5022
\Model : CatBoost
Best recall Score : 0.431
Test recall : 0.472
SAMPLING : ADASYN
\Model : Decision Tree
Best recall Score : 0.5919
Test recall : 0.6153
\Model : Random Forest
Best recall Score : 0.5722
Test recall : 0.6056
\Model : KNN
Best recall Score : 0.5426
Test recall : 0.57
\Model : Logistic Regression


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Best recall Score : 0.458


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Test recall : 0.4666
\Model : XGBoost
Best recall Score : 0.4588
Test recall : 0.4731
\Model : CatBoost
Best recall Score : 0.42
Test recall : 0.4504


In [74]:
#Friedman test
print("FRIEDMAN TEST")
stat, p = friedmanchisquare(*friedman_scores.values())
print("Statistic :", stat)
print("P-value :", p)

#Rank Calculation
scores_array = np.array(list(friedman_scores.values())).T

rank_matrix = []

for row in scores_array:

    ranks = rankdata(-row)

    rank_matrix.append(ranks)

rank_matrix = np.array(rank_matrix)

mean_ranks = rank_matrix.mean(axis=0)

#comparison Dataframe
friedman_df = pd.DataFrame({

    'Model': list(friedman_scores.keys()),

    'Mean Rank': np.round(mean_ranks, 3),

    'Mean CV recall Score': np.round(
        list(friedman_mean_scores.values()), 4
    )
})

friedman_df = friedman_df.sort_values(
    by='Mean Rank'
)

summary_df = pd.DataFrame({

    'Statistic': [round(stat, 4)],

    'P-value': [round(p, 6)],

    'Significant': ['Yes' if p < 0.05 else 'No']
})

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by='Recall',
    ascending=False
)

FRIEDMAN TEST
Statistic : 82.75345860004128
P-value : 1.238796042889758e-10


In [82]:
  with pd.ExcelWriter("All_models_result.xlsx") as writer:
    # ALL MODEL RESULTS
    results_df.to_excel(
        writer,
        sheet_name='Model Results',
        index=False
    )
  with pd.ExcelWriter("Friedman Rankings.xlsx") as writer:
    # FRIEDMAN COMPARISON(ranking)
    friedman_df.to_excel(
        writer,
        sheet_name='Friedman Rankings',
        index=False
    )
  with pd.ExcelWriter("ML_Comparison_Results.xlsx") as writer:
    # FRIEDMAN SUMMARY
    summary_df.to_excel(
        writer,
        sheet_name='Friedman Test',
        index=False
    )